In [2]:
import pandas as pd
import sqlite3
import joblib
import json
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

# 1. Load Data
conn = sqlite3.connect('../data/warehouse.db')
query = "SELECT subject, body, priority, tag_1 AS queue FROM tickets"
df = pd.read_sql_query(query, conn)
conn.close()

# Drop missing values in targets or text
df = df.dropna(subset=['subject', 'body', 'queue', 'priority'])

# Combine subject and body as the main text feature
df['text'] = df['subject'] + " " + df['body']

# FILTER: Keep only queues with at least 2 instances so stratify works
valid_queues = df['queue'].value_counts()[df['queue'].value_counts() > 1].index
df = df[df['queue'].isin(valid_queues)]

# 2. Train/Test Split
X_train, X_test, y_queue_train, y_queue_test, y_prio_train, y_prio_test = train_test_split(
    df['text'], df['queue'], df['priority'], test_size=0.2, random_state=42, stratify=df['queue']
)

# 3. Majority-Class Baseline for Queue
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_queue_train)
dummy_queue_preds = dummy_clf.predict(X_test)

dummy_acc = accuracy_score(y_queue_test, dummy_queue_preds)
dummy_f1 = f1_score(y_queue_test, dummy_queue_preds, average='macro')

# 4. TF-IDF + Logistic Regression for Queue
queue_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

queue_pipeline.fit(X_train, y_queue_train)
queue_preds = queue_pipeline.predict(X_test)

queue_acc = accuracy_score(y_queue_test, queue_preds)
queue_f1 = f1_score(y_queue_test, queue_preds, average='macro')

# 5. TF-IDF + Logistic Regression for Priority
prio_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

prio_pipeline.fit(X_train, y_prio_train)
prio_preds = prio_pipeline.predict(X_test)

prio_acc = accuracy_score(y_prio_test, prio_preds)
prio_f1 = f1_score(y_prio_test, prio_preds, average='macro')

# 6. Leaderboard & Results
print("--- TRIAGE LEADERBOARD ---")
print(f"Queue Baseline (Majority Class) - Acc: {dummy_acc:.4f} | Macro-F1: {dummy_f1:.4f}")
print(f"Queue Model (TF-IDF + LR)       - Acc: {queue_acc:.4f} | Macro-F1: {queue_f1:.4f}")
print(f"Priority Model (TF-IDF + LR)    - Acc: {prio_acc:.4f} | Macro-F1: {prio_f1:.4f}")

# 7. Save Models and Metadata
joblib.dump(queue_pipeline, '../models/tfidf_queue_model.joblib')
joblib.dump(prio_pipeline, '../models/tfidf_priority_model.joblib')

metadata = {
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "queue_accuracy": queue_acc,
    "queue_macro_f1": queue_f1,
    "priority_accuracy": prio_acc,
    "priority_macro_f1": prio_f1
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)
    
print("\nModels and metadata successfully saved to the /models directory.")

--- TRIAGE LEADERBOARD ---
Queue Baseline (Majority Class) - Acc: 0.2721 | Macro-F1: 0.0097
Queue Model (TF-IDF + LR)       - Acc: 0.6777 | Macro-F1: 0.4396
Priority Model (TF-IDF + LR)    - Acc: 0.4677 | Macro-F1: 0.4527

Models and metadata successfully saved to the /models directory.
